In [5]:
import sys
from pathlib import Path

PROJECT_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "analysis").is_dir())
sys.path.insert(0, str(PROJECT_ROOT))

import os
from dotenv import load_dotenv, find_dotenv
import xarray as xr

import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from datetime import date

import analysis.analysis_utils as analysis_utils
import analysis.isku_utils as isku_utils

import importlib

importlib.reload(analysis_utils)
importlib.reload(isku_utils)

<module 'analysis.isku_utils' from '/home/emily_zuetell/projects/poreallas/analysis/isku_utils.py'>

In [11]:
load_dotenv(find_dotenv("october_local.env", raise_error_if_not_found=True), override=True)

# Define Forecast
EFFECTS_URI = os.environ["POREALLAS_EFFECTS_URI"]

In [12]:
RUN_MONTH = "2026-10"
N_LEADS = 6

_months = pd.period_range(pd.Period(RUN_MONTH, "M"), periods=N_LEADS, freq="M")
FC_MONTHS = [p.month for p in _months]
FC_PERIOD = slice(_months[0].start_time.strftime("%Y-%m-%d"),
                  _months[-1].end_time.strftime("%Y-%m-%d"))

# Baseline period for Impact
BASELINE_PERIOD = slice("1996-01-01", "2025-12-31")


config = analysis_utils.ImpactConfig( version = f"v{date.today():%y%m%d}",
                                     baseline_period=BASELINE_PERIOD, 
                                     polygons_path= os.environ["POREALLAS_REGIONS_POLYGONS_URI"],
                                     socioeconomics_path= os.environ["POREALLAS_SOCIOECONOMICS_URI"],
                                     rate=False, 
                                     months = FC_MONTHS,
                                     hotonly = "hotonly", 
                                     dims = ['number', 'sample'])

In [13]:
# Projection Effects
effect = xr.open_datatree(EFFECTS_URI, consolidated=False)

In [15]:
### Log baseline period and impact calculation
rate_l = "rate" if config.rate else "total"
baseline_tag = analysis_utils._baseline_tag(config.baseline_period)

In [16]:
# Compute impact: forecast - baseline
impact = analysis_utils.compute_impact(
    effect.chunk({dim: -1 for dim in config.dims}),
    config,
    ensemble=True,
)
# Use only the defined 6-months
impact = impact.sel(month=config.months)

In [17]:
# Aggregate Impact Regions to group_level
group_level = 'ISO' #IR: Impact region, # ADM1: State level, # ISO: Country level
# Use a subset of ensemble members for quick testing (.sel(number = ...))
impact, merge_key, base_cols = analysis_utils.aggregate_impact(impact, config, group_level)

In [22]:
impact.mean(dim = ['number', 'sample']).sum(dim = 'month').sum(dim = 'ISO').values

/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)


array(372932.70021184)

In [24]:

impact_sahel = impact.sel(ISO = ['NGA', 'SDN', 'NER', 'TCD'])
impact_seasia = impact.sel(ISO = ["PHL", "VNM", "THA", "KHM"])

In [25]:
def crossed_sem(da, a="number", b="sample", include_resid = False):
    # Crossed standard error
    # Size of dims (number, sample)
    na, nb = da.sizes[a], da.sizes[b]
    # grand mean
    grand = da.mean((a, b))
    m_a = da.mean(b)                     # per-member means (over samples)
    m_b = da.mean(a)                     # per-sample means (over members)

    resid = da - m_a - m_b + grand       # interaction/residual term
    if include_resid:
        var_e = (resid**2).sum((a, b)) / ((na - 1) * (nb - 1))
    else:
        var_e = 0
    
    var_mean = (m_a.var(a, ddof=1) / na
                + m_b.var(b, ddof=1) / nb
                - var_e / (na * nb))
    return grand, np.sqrt(var_mean.clip(min=0))

In [26]:
mean_sahel, se_sahel = crossed_sem(impact_sahel.sum(dim = ['month', 'ISO']), a="number", b="sample", include_resid = True)

In [27]:
mean_seasia, se_seasia = crossed_sem(impact_seasia.sum(dim = ['month', 'ISO']), a="number", b="sample", include_resid = True)

In [28]:
print(f"Sahel: {mean_sahel.values} +/- {1.96*se_sahel.values}")
print(f"SE Asia: {mean_seasia.values} +/- {1.96*se_seasia.values}")

/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)


Sahel: 73018.13563076583 +/- 9760.306034374238


/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)


SE Asia: 23792.5485974844 +/- 3922.279409008067


In [28]:
mean, se = crossed_sem(impact.sum(dim = 'month'), a="number", b="sample", include_resid = True)
mean, se_no_resid = crossed_sem(impact.sum(dim = 'month'), a="number", b="sample", include_resid = False)

In [19]:
#Compute stats in xarray from dims in config.dims
stat_cols = ["median", "p17", "p83", "likely_range_IPCC", "mean", "std", "min", "max", "p10", "p90"]
_polygons_impact = analysis_utils.dataset_to_dataframe(analysis_utils.compute_stats(impact, dim=config.dims))

/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)
/home/emily_zuetell/projects/poreallas/.venv/lib/python3.14/site-packages/dask/_task_spec.py:768: RuntimeWarning: invalid value encountered in divide
  return self.func(*new_argspec)


KeyboardInterrupt: 

In [54]:
filename_template="{version}_{hotonly}_{scope}_{rate_l}_{stat_scope}_{group_level}_{baseline}_{cleaned}.csv"
# "regional_monthly" 
_polygons_impact = analysis_utils.dataset_to_dataframe(analysis_utils.compute_stats(impact, dim=config.dims))
wide = _polygons_impact.pivot(
    index=base_cols,
    columns="month", values=stat_cols,
)
wide.columns = [f"month {m} {stat}" for stat, m in wide.columns]
stat_col_names = wide.columns.difference(base_cols)
wide = wide.reset_index()
wide_rounded = analysis_utils.round_output(wide)
wide.to_csv(
    filename_template.format(
        version=config.version,
        hotonly=config.hotonly,
        rate_l=rate_l,
        scope="monthly",
        stat_scope="",
        group_level=group_level,
        baseline=baseline_tag,
        cleaned = 'raw'
    ),
    index=False,
)
wide_rounded.to_csv(
    filename_template.format(
        version=config.version,
        hotonly=config.hotonly,
        rate_l=rate_l,
        scope="monthly",
        stat_scope="",
        group_level=group_level,
        baseline=baseline_tag,
        cleaned = 'rounded'
    ),
    index=False,
)
